# LangGraph

### LangChain vs LangGraph

**核心定位：**

- **LangGraph** 是底层编排引擎（Orchestration Layer）。 它提供通用的**图（Graph）**机制，支持任意复杂的流程：线性、分支、**循环**、多路径、持久化状态、Human-in-the-Loop 等。
- **LangChain** 是建立在 LangGraph 之上的**高级应用框架**。 它提供了大量现成的“积木”：Models、Tools、Prompts、Retrievers、Memory、Chains、Agents 等，让你能快速搭建应用。
- LangChain 中的很多“单执行流”实现（例如传统的 LCEL Chain、旧版 AgentExecutor），本质上可以看作是 **LangGraph 的一个特殊情况** —— 即**没有循环、没有分支的线性图（DAG）**。

**形象比喻（优化版）：**

- **LangChain** ≈ 乐高积木盒 + 一些**预装好的简单模型**（比如一辆已经组装好的小汽车半成品）。 你可以直接拿来玩，快速上手做出能跑的东西。
- **LangGraph** ≈ 乐高积木的**“搭建说明书 + 底层连接系统”**（各种连接块、齿轮、转轴、按钮）。 它让你可以把积木拆开，按照自己的想法拼成任意复杂的机器 —— 可以有循环齿轮、条件分支路径、需要人工按按钮的交互部分，甚至多条并行流水线。

**最常见的使用方式：**

有两种玩法：

1. **快速模式**：直接使用 LangChain 提供的预装模型（例如 create_agent），几行代码就能跑起来一个简单 Agent。
2. **进阶模式**：把 LangChain 提供的零件（model、tools、prompt 等）拆出来，放到 LangGraph 的 StateGraph 中，自己定义节点和边的连接方式，从而实现循环、条件路由、人工审核、多 Agent 协作等复杂逻辑。

**生产实践中的黄金组合**：

> **LangChain 负责提供“高质量的零件”和“简单组装”** **LangGraph 负责“整体流程编排”和“复杂逻辑控制”**

在实际开发中，你通常会：

- 用 LangChain 创建 model、tools、prompt
- 把它们塞进 LangGraph 的各个 Node 里
- 用 LangGraph 的 StateGraph 定义节点之间的连接、循环、条件路由、中断等

这正是我们前面一系列例子（条件路由、ReAct Agent 节点、自我纠错循环、Human-in-the-Loop）的共同模式。



## 实战（From shallow to deep）

### Hello LangGraph

#### 代码案例

这个例子展示：

如何使用 StateGraph 定义一个只有一个节点的简单图（Graph），节点中调用了一个模拟的 LLM 函数，最终返回一条固定的 "hello world!" 回复。通过这个例子，你可以快速理解 LangGraph 的核心概念：状态（MessagesState）、节点（Node）、边（Edge）以及图的编译与调用。

In [4]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage, AIMessage

def mock_llm(state: MessagesState):
    return {"messages": [
        AIMessage("hello world!")
    ]}


graph = StateGraph(MessagesState)
graph.add_node(mock_llm)
graph.add_edge(START, "mock_llm")
graph.add_edge("mock_llm", END)

graph = graph.compile()

graph.invoke({"messages": [
    HumanMessage(content="Hi!")
]})

{'messages': [HumanMessage(content='Hi!', additional_kwargs={}, response_metadata={}, id='02ee3ee4-9a35-47e7-bd5a-2739c8677248'),
  AIMessage(content='hello world!', additional_kwargs={}, response_metadata={}, id='19f44256-a755-437b-badb-9dff54a75a7c', tool_calls=[], invalid_tool_calls=[])]}

#### 代码解析

#### **(1) 定义状态：`MessagesState`**

```Python
graph = StateGraph(MessagesState)
```

- **作用**：定义“工厂”里流动的**标准容器**。
- **细节**：`MessagesState` 是 LangGraph 内置的一个结构，它本质上是一个字典，里面包含一个 `messages` 列表。
- **魔法点**：它内置了一个 **`Annotated[list, add_messages]`** 逻辑。这意味着当你返回新的消息时，它不是覆盖旧消息，而是**追加（Append）**到列表末尾。

#### **(2) 定义节点（Node）：`mock_llm`**

```Python
def mock_llm(state: MessagesState):
    return {"messages": [{"role": "ai", "content": "hello world"}]}
```

- **作用**：工厂里的**加工车间**。
- **细节**：
  - **输入**：当前时刻的完整“状态”（即已经产生的对话历史）。
  - **输出**：它**只返回它想更新的部分**。这里它返回了一条 AI 消息，这会被自动追加到 State 的 `messages` 列表里。

#### **(3) 构建图结构（Edges）：`add_edge`**

```Python
graph.add_node(mock_llm) # 注册车间
graph.add_edge(START, "mock_llm") # 入口 -> 车间
graph.add_edge("mock_llm", END)   # 车间 -> 出口
```

- **作用**：铺设**传送带**。
- **细节**：
  - `START` 和 `END` 是特殊的虚拟节点。
  - 这定义了一个极其简单的线性流：**开始 -> 执行模拟模型 -> 结束**。

#### **(4) 编译与触发：`compile` & `invoke`**

```Python
graph = graph.compile() # 将蓝图转换成可运行的程序
graph.invoke({"messages": [{"role": "user", "content": "hi!"}]})
```

- **作用**：启动生产线。
- **细节**：`invoke` 传入的是**初始状态**。Graph 会从 `START` 开始，沿着你定义的 `edge` 走完所有流程，最后返回最终的完整 `State`。

### Conditional Router（条件分支与路由）



#### 代码案例

这是一个 LangGraph **条件路由（Conditional Edge）** 的经典入门示例。 它展示了如何根据用户输入的内容动态决定执行哪个节点：

- 如果用户消息中包含“天气”关键词，则路由到 weather_node
- 否则路由到普通 chat_node

通过这个例子，你可以清晰理解 LangGraph 中 **add_conditional_edges** 的使用方式，包括路由函数的写法、路由映射表的定义，以及如何让图根据不同条件走不同的分支路径。

In [ ]:
from typing import Literal
# --- 1. 定义节点逻辑 ---

def chatbot(state: MessagesState):
    """普通对话节点"""
    return {"messages": [AIMessage(content="这是一条普通的 AI 回复。")]}

def weather_node(state: MessagesState):
    """专门处理天气的节点"""
    return {"messages": [AIMessage(content="查询结果：上海今天晴，25°C。")]}

# --- 2. 定义条件路由逻辑 ---

def router_logic(state: MessagesState) -> Literal["weather", "chat"]:
    """
    这就是“分路器”：检查最后一条消息的内容。
    返回字符串必须匹配后面 add_conditional_edges 中定义的 key。
    """
    last_message = state["messages"][-1].content
    if "天气" in last_message:
        return "weather"
    
    return "chat"

# --- 3. 构建工作流图 ---

workflow = StateGraph(MessagesState)

# 添加节点
workflow.add_node("chat_node", chatbot)
workflow.add_node("weather_node", weather_node)

# 设置入口后的“条件路由”
workflow.add_conditional_edges(
    START,           # 从开始节点后出发
    router_logic,    # 使用这个函数决定去向
    {                # 路由映射表：函数返回什么值，就走哪个节点
        "weather": "weather_node",
        "chat": "chat_node"
    }
)

# 无论走哪个节点，最后都指向结束
workflow.add_edge("chat_node", END)
workflow.add_edge("weather_node", END)
# 编译
app = workflow.compile()

# --- 4. 测试不同路径 ---

print("--- 测试 1: 问天气 ---")
res1 = app.invoke({"messages": [HumanMessage(content="帮我查查天气")]})
print(res1["messages"][-1].content)

print("\n--- 测试 2: 闲聊 ---")
res2 = app.invoke({"messages": [HumanMessage(content="你好呀")]})
print(res2["messages"][-1].content)

--- 测试 1: 问天气 ---
查询结果：上海今天晴，25°C。

--- 测试 2: 闲聊 ---
这是一条普通的 AI 回复。


#### 代码解析

**`add_conditional_edges` (条件边)**：

- 它是 LangGraph 的灵魂。普通的 `add_edge` 是硬连接，而它是**软连接**。
- 它的添加使得具备了**“分类分诊”**的能力，不再是收到什么都一股脑传给模型。

**`Literal` 标注**：

- 使用 `Literal["weather", "chat"]` 这种类型提示是一个好习惯。它明确告诉图结构：这个路由函数只会输出这两种结果，增强了代码的稳健性。

**状态驱动 (State-Driven)**：

- 注意 `router_logic` 函数。它不需要调用 LLM，只需要简单的 Python 逻辑（`if "天气" in last_message`）就能控制复杂的流程。
- **省钱/提速**：在实际工程中，很多简单的路由（比如判断用户是否说“再见”）可以用正则实现，没必要每次都问大模型。

### ReAct Agent Router

#### 代码案例

这是一个 **LangGraph 中集成 LangChain Agent 的条件路由示例**。 它展示了如何：

- 使用 create_agent 创建带工具（get_weather）的 Agent；
- 将 Agent 包装成一个节点（agent_worker），并通过 add_conditional_edges 实现智能路由；
- 当用户询问天气相关内容时，路由到 Agent 节点（让 Agent 自主决定是否调用工具），否则走普通闲聊节点。

In [7]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool


# 0. 封装好Agent，方便后续调用
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."


model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)
webster_agent = create_agent(
    model,
    tools=[get_weather],
    name="webster_agent"
)

# 1. 定义调用 Agent 节点逻辑
def webster_agent_node(state: MessagesState):
    """
    调用真正的 Webster Agent 节点。
    直接把整个 state 传入 agent，让 agent 自行决定是否调用工具、思考、输出。
    """
    # 真实调用 agent（推荐方式）
    result = webster_agent.invoke(state)
    
    # result 通常包含 "messages"，我们直接返回更新后的 messages
    return {"messages": result["messages"]}


# --- 2. 定义普通对话节点 ---
def general_chat_node(state: MessagesState):
    return {"messages": [AIMessage(content="你好！我是 Webster 的闲聊模式。")]}

# --- 3. 路由逻辑 ---
def router(state: MessagesState) -> Literal["call_agent", "just_chat"]:
    last_msg = state["messages"][-1].content.lower()
    if any(kw in last_msg for kw in ["查", "天气", "weather"]):
        return "call_agent"
    return "just_chat"

# --- 4. 构建图 ---
workflow = StateGraph(MessagesState)
# 添加节点
workflow.add_node("agent_worker", webster_agent_node)
workflow.add_node("chat_worker", general_chat_node)

# 设置路由
workflow.add_conditional_edges(
    START,
    router,
    {
        "call_agent": "agent_worker",
        "just_chat": "chat_worker"
    }
)
# 连接到结束
workflow.add_edge("agent_worker", END)
workflow.add_edge("chat_worker", END)
app = workflow.compile()

# --- 5. 测试 ---# --- 6. 测试 ---
print("--- 测试：调用 Agent 查天气 ---")
result1 = app.invoke({"messages": [HumanMessage(content="帮我查查上海天气")]})
print(result1["messages"][-1].content)

print("\n--- 测试：普通闲聊 ---")
result2 = app.invoke({"messages": [HumanMessage(content="你好呀，今天怎么样？")]})
print(result2["messages"][-1].content)

--- 测试：调用 Agent 查天气 ---
目前上海的天气是晴朗的。

--- 测试：普通闲聊 ---
你好！我是 Webster 的闲聊模式。


#### 代码解析

##### ReAct

> ### ReAct 到底是什么？
>
> **ReAct** 是 **Reasoning + Acting** 的缩写，中文常译为 **“推理 + 行动”**。
>
> 它是一种让大语言模型（LLM）像“思考的人”一样工作的经典方法，而不是直接一次性给出答案。
>
> #### ReAct 的核心循环（Loop）非常简单，只有三步，不断重复：
>
> 1. **Reason（推理 / 思考）** 模型先思考：“现在的问题是什么？我需要做什么？目前我知道什么？还缺什么信息？”
> 2. **Act（行动）** 模型决定采取行动 —— 通常是**调用工具**（比如查天气、搜索网页、计算等）。 在你的例子中，就是调用 get_weather 工具。
> 3. **Observe（观察 / 反馈）** 模型拿到工具返回的结果（Observation），然后把这个结果当成新信息，继续下一步思考。
>
> → 重复 **Reason → Act → Observe**，直到模型认为信息足够，可以给出最终答案，就停止循环，直接回复用户。
>
> #### 举个直观的例子（用你的天气查询）：
>
> 用户说：“帮我查查上海天气”
>
> - **Reason**：模型思考 → “我不知道上海现在的天气，我有 get_weather 这个工具，应该用它。”
> - **Act**：模型调用工具 → get_weather("上海")
> - **Observe**：工具返回 → “It's sunny in 上海.”
> - **Reason**：模型思考 → “现在我知道了，上海是晴天，我可以直接回答用户了。”
> - **最终输出**： “目前上海的天气是晴朗的。”
>
> 这就是一个完整的 ReAct 过程。如果问题更复杂（比如“上海今天天气怎么样？适合穿什么衣服？”），它可能会多循环几次：先查天气，再根据天气推理穿衣建议。



##### 为什么要把 Agent 放进 Node 里？

1. **责任分离**：你可以让一个 Node 只负责写代码，另一个 Node 只负责跑测试。即使它们内部都是复杂的 Agent，但对 Graph 来说，它们只是任务块。
2. **状态共享**：Agent 节点运行完后，它的思考过程和结果都会存在 `MessagesState` 里，下一个节点能直接看到。
3. **为“循环”做准备**：这是最关键的。如果 Agent 跑出来的结果不对，Graph 可以让数据**流回**这个节点重跑。



### Self-Correction Loop 自我纠错循环

#### 代码示例

这是一个 **LangGraph 自我纠错 / 学习循环** 的入门示例。 它实现了“生成 → 批判 → 如果不满意就回炉重造”的闭环：

- generator 节点先产生答案
- critic 节点像审稿人一样评估答案质量
- 如果 critic 认为需要改进，就**自动路由回 generator**（带上之前的批判反馈），让它自我修订
- 直到 critic 给出「ACCEPT」才结束

这是从「单次路由」到「带反馈的循环工作流」的自然进阶，非常适合理解 LangGraph 中**如何用条件边实现循环**以及 Agent 的自我学习能力。

In [9]:
# 0. 初始化模型
model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.3   # 稍微提高一点创造性，但保持稳定
)

# --- 1. 生成器节点（负责产生初始/修订答案）---
def generator(state: MessagesState):
    # 把整个历史消息传给模型，让它知道之前的批判
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}

# --- 2. 批判器节点（审稿人，决定是否接受或要求修改）---
def critic(state: MessagesState):
    # 构造专门的批判 prompt
    critique_prompt = [
        HumanMessage(content="""你是一个严格的审稿人。
请评估最后一条 AI 回复的质量：
- 检查事实是否准确、逻辑是否清晰、表达是否流畅
- 如果满意，请在最后一行严格输出：Decision: ACCEPT
- 如果需要改进，请给出具体改进建议，并在最后一行输出：Decision: REVISE

请直接开始评估，不要说多余的话。""")
    ]
    
    # 把历史 + 批判 prompt 拼在一起
    full_messages = state["messages"] + critique_prompt
    response = model.invoke(full_messages)
    return {"messages": [response]}

# --- 3. 路由逻辑（决定是继续循环还是结束）---
def self_correction_router(state: MessagesState) -> Literal["revise", "accept"]:
    last_message = state["messages"][-1].content.lower()
    
    # 安全机制：最多循环 5 次（防止死循环）
    ai_messages_count = sum(1 for m in state["messages"] if isinstance(m, AIMessage))
    if ai_messages_count > 10:
        return "accept"
    
    if "decision: accept" in last_message:
        return "accept"
    else:
        return "revise"
    
# --- 4. 构建工作流图 ---
workflow = StateGraph(MessagesState)

# 添加节点
workflow.add_node("generator", generator)
workflow.add_node("critic", critic)

# 设置流程
workflow.add_edge(START, "generator")           # 开始 → 生成答案
workflow.add_edge("generator", "critic")        # 生成 → 批判评估

# 关键：从 critic 出发的条件路由（实现循环！）
workflow.add_conditional_edges(
    "critic",
    self_correction_router,
    {
        "revise": "generator",   # 继续循环：回 generator 修订
        "accept": END            # 结束：输出最终答案
    }
)
# 编译
app = workflow.compile()

# --- 5. 测试 ---
print("=== 测试自我纠错循环 ===")
result = app.invoke({
    "messages": [HumanMessage(content="请计算 15 × 27 的结果，并详细解释计算过程。")]
})

print("\n最终输出：")
print(result["messages"][-1].content)

=== 测试自我纠错循环 ===

最终输出：
Decision: ACCEPT

回复内容准确地解答了问题，逻辑清晰，表达流畅。


#### 代码解析

**如何实现“循环”（这是本例最大亮点）** 

之前例子里的边都是单向（START → node → END）。 本例通过 add_conditional_edges 把 critic 节点的输出路由回 generator，就形成了**闭环**：

```Python
workflow.add_conditional_edges(
    "critic",
    self_correction_router,
    {"revise": "generator", "accept": END}
)
```

这是 LangGraph 中实现自我纠错/迭代优化的**标准写法**。

**critic 节点（新角色：审稿人）** 

它不是简单回复，而是专门“挑刺”：

- 接收完整历史消息（包含之前的 generator 输出和上一次批判）
- 要求模型**严格在最后一行输出 Decision: ACCEPT 或 Decision: REVISE**（结构化输出） 这样 router 就能轻松判断是否继续循环。

**self_correction_router（新路由逻辑）**

- 不再只看关键词，而是看 critic 最后输出的 Decision:
- 新增**安全计数器**（ai_messages_count）：防止模型一直改不完导致死循环（实际生产中非常推荐）
- 返回 "revise"（回 generator）或 "accept"（直接 END）

**MessagesState 的“学习”能力** 

每次 generator 和 critic 都会把消息 append 到同一个 state["messages"] 里。 下一次 generator 看到的是： 用户问题 → 第1版答案 → 第1次批判 → 第2版答案 → 第2次批判 …… 这就是**自我学习循环**的核心：Agent 能“记住”自己之前的错误并改进。

**与之前例子的区别**

- 之前是“一次路由选分支”，这次是“多次循环自我迭代”
- 没有用到 ReAct Agent（故意简化），让你先掌握纯 LangGraph 的循环机制
- 之后你可以轻松把 generator 换成 create_react_agent，实现“带工具的自我纠错 Agent”

### Human-in-the-Loop（人工介入循环）

#### 代码案例

这是一个**终端实时交互**的 Human-in-the-Loop 示例。 AI 先使用 ReAct Agent 生成回复，随后进入人工审核节点打印答案并暂停等待用户决策。用户可输入 yes/批准/good 等表示满意直接结束，或输入修改意见让 Agent 重新生成。该版本采用推荐的 stream + get_state 判断中断方式，稳定性更高。

**学习收获：** 掌握 LangGraph 中实现真正人机交互循环的标准写法，包括如何安全检测中断和恢复执行。

- 使用 MemorySaver 作为检查点（Checkpoint），确保中断后状态能正确恢复
- 在 human_review 节点中使用 interrupt() 暂停执行，等待用户输入。
- 通过 Command(resume=...) 恢复图的运行
- 适合需要人工把关的场景（如天气建议、敏感回复等）

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# ====================== 工具 & Agent ======================
@tool
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    return f"It's sunny in {location}."

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M",
    model_provider="ollama",
    temperature=0.2
)

webster_agent = create_react_agent(model=model, tools=[get_weather])

# ====================== 节点 ======================
def agent_worker(state: MessagesState):
    result = webster_agent.invoke(state)
    return {"messages": result["messages"]}

def human_review_node(state: MessagesState):
    """人工审核节点"""
    last_ai_content = state["messages"][-1].content
    
    print("\n" + "="*70)
    print("🤖 AI 生成的回复：")
    print(last_ai_content)
    print("="*70)
    
    # interrupt 会暂停图执行（不要 try/except 包裹它！）
    feedback = interrupt({
        "ai_response": last_ai_content,
        "instruction": "请输入决策：\n- 'yes' / '批准' / 'ok' 表示满意并结束\n- 否则输入修改意见，让 Agent 重新生成"
    })
    
    return {"messages": [HumanMessage(content=f"Human feedback: {feedback}")]}

def review_router(state: MessagesState) -> Literal["revise", "accept"]:
    last_content = state["messages"][-1].content.lower()
    if any(k in last_content for k in ["yes", "批准", "ok", "good", "满意", "可以"]):
        return "accept"
    return "revise"

# ====================== 构建图 ======================
workflow = StateGraph(MessagesState)
workflow.add_node("agent_worker", agent_worker)
workflow.add_node("human_review", human_review_node)

workflow.add_edge(START, "agent_worker")
workflow.add_edge("agent_worker", "human_review")

workflow.add_conditional_edges(
    "human_review",
    review_router,
    {"revise": "agent_worker", "accept": END}
)

checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

# ====================== 交互主循环（最推荐写法） ======================
print("=== LangGraph Human-in-the-Loop 交互演示 v3 ===")
print("问题：帮我查查上海的天气，并给出出行建议\n")

thread_config = {"configurable": {"thread_id": "hil_weather_v3"}}

# 初始输入
inputs = {"messages": [HumanMessage(content="帮我查查上海的天气，并给出出行建议")]}

while True:
    # 使用 stream + interrupt 检查（更稳定）
    for event in app.stream(inputs, config=thread_config, stream_mode="values"):
        # 可以在这里打印中间状态（可选）
        pass

    # 检查当前状态是否还在等待 interrupt
    current_state = app.get_state(thread_config)
    
    if current_state.next:  # 有 pending node → 说明触发了 interrupt
        user_input = input("\n你的决策 >>> ").strip()
        if not user_input:
            user_input = "yes"
        
        # 恢复执行
        inputs = Command(resume=user_input)
        continue
    else:
        # 没有 pending node → 已到达 END
        print("\n✅ 流程结束！最终回复：")
        print(current_state.values["messages"][-1].content)
        break

=== LangGraph Human-in-the-Loop 交互演示 ===
用户问题：帮我查查上海的天气，并给出出行建议


🤖 AI 生成的回复：
content='目前上海是晴天。根据当前的天气情况，您可以正常外出活动。不过还是建议您关注一下最新的天气预报，以便做好充分准备。如果有更多需要帮助的地方，请告诉我！' additional_kwargs={} response_metadata={'model': 'qwen2.5:7b-instruct-q4_K_M', 'created_at': '2026-04-16T14:21:48.9899969Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2832413400, 'load_duration': 121756200, 'prompt_eval_count': 198, 'prompt_eval_duration': 229805300, 'eval_count': 41, 'eval_duration': 2401940900, 'logprobs': None, 'model_name': 'qwen2.5:7b-instruct-q4_K_M', 'model_provider': 'ollama'} id='lc_run--019d96ab-5d2b-77e2-886f-b22190c14d5f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 198, 'output_tokens': 41, 'total_tokens': 239}

✅ 流程已结束，最终回复如下：
目前上海是晴天。根据当前的天气情况，您可以正常外出活动。不过还是建议您关注一下最新的天气预报，以便做好充分准备。如果有更多需要帮助的地方，请告诉我！


#### 代码解析

> **不需要** HumanInTheLoopMiddleware（那是 LangChain 旧版 Agent 的 middleware，主要用于 create_agent + 工具级中断审批）。
>
> 现在用的是**纯 LangGraph 自定义节点 + create_react_agent**，正确的做法是使用 **interrupt() + Command(resume=...)** + **checkpointer**。
>
> ### 1. HumanInTheLoopMiddleware 到底是什么？能不能用于 LangGraph？
>
> **正确答案**：
>
> - **HumanInTheLoopMiddleware** 是 **LangChain** 提供的**中间件（middleware）**，专门为 from langchain.agents import create_agent 这个新版 Agent 创建函数设计的。
> - 它的作用是**在工具调用（tool call）层面**实现人工审核：当 Agent 准备调用某个工具时（比如发邮件、写文件、执行危险操作），可以暂停、让人类 approve / edit / reject。
> - 它**底层依赖 LangGraph 的 interrupt 机制**，但它**只能用在 create_agent 创建的 Agent 上**。
> - **不能直接用于**你自己用 StateGraph 手动搭建的纯 LangGraph 工作流（比如我们前面写的 human_review_node）。
>
> **总结**：
>
> - 想做**工具级**（tool-level）中断审批 → 用 HumanInTheLoopMiddleware + create_agent（最简单）。
> - 想做**任意节点级**（node-level）人工审核（如我们之前的天气审核例子）→ 直接在 LangGraph 里用 interrupt() + 自定义节点（这就是我们 v3 版本的做法）。
>
> 所以你之前的理解是正确的：它主要服务于“LangChain 新版 Agent”的工具中断场景，不是通用 LangGraph 的 HITL 方案。
>
> ### 2. create_react_agent 和 create_agent 的区别？为什么 VSCode 有删除线？
>
> **最新情况（2025 年 10 月 LangGraph v1.0 后）**：
>
> - langgraph.prebuilt.create_react_agent **已被官方标记为 Deprecated（废弃）**。
>
> - VSCode 上的删除线就是这个原因。官方迁移提示是：
>
>   > “create_react_agent has been moved to langchain.agents. Please update your import to from langchain.agents import create_agent.”
>
> **两者区别**（对象层面 + 概念层面）：
>
> | 维度     | create_react_agent（旧，已废弃） | create_agent（新，推荐）                                     |
> | -------- | -------------------------------- | ------------------------------------------------------------ |
> | 所在包   | langgraph.prebuilt               | langchain.agents                                             |
> | 本质     | LangGraph 预置的简单 ReAct 图    | LangChain 官方新的 Agent 工厂函数（内部也是 LangGraph）      |
> | 功能     | 只能做经典 ReAct 循环，功能较少  | 支持 **middleware 系统**（包括 HumanInTheLoopMiddleware）、更灵活的 prompt、多种 Agent 类型 |
> | 扩展性   | 较弱                             | 强（可轻松插入自定义中间件、修改状态等）                     |
> | 状态管理 | 基本 MessagesState               | 更强大，支持自定义 state schema                              |
> | 未来支持 | 即将移除（v2.0）                 | 官方主推，持续更新                                           |
>
> **一句话总结**：
>
> - 以前我们用 create_react_agent 是为了快速演示 ReAct 循环。
> - 现在官方统一推荐 create_agent，因为它**本质上就是升级版的 ReAct Agent**，而且集成了 middleware 系统（这也是为什么 HumanInTheLoopMiddleware 只能搭配它使用）。
>
> 以后写新代码时，建议直接改成：
>
> ```Python
> from langchain.agents import create_agent
> agent = create_agent(model=model, tools=[get_weather])
> ```



**interrupt() 函数（Human-in-the-Loop 的核心机制）**

```Python
feedback = interrupt({
    "ai_response": last_ai_content,
    "instruction": "请输入决策：..."
})
```

- 作用：在节点执行到这里时**主动暂停整个图的运行**，并把字典中的信息抛给外部调用者。
- 当外部使用 Command(resume=xxx) 恢复时，feedback 会接收到用户输入的内容。
- interrupt() 在节点内部会抛出一个特殊的 GraphInterrupt 异常（而不是像 input() 那样直接阻塞）。
- **注意**：不能在 interrupt() 外面加 try/except，否则会吞掉中断，导致无法暂停。

**app.stream(inputs, config, stream_mode="values")**

```Python
for event in app.stream(inputs, config=thread_config, stream_mode="values"):
    pass
```

- 这是 LangGraph 推荐的运行方式，尤其在需要处理 interrupt 的场景。
- stream_mode="values" 表示每次返回完整的 state 值。
- 即使图因为 interrupt 而暂停，stream 也会正常结束本次循环，让我们有机会检查状态。

**app.get_state(thread_config) 检查中断状态**

```Python
current_state = app.get_state(thread_config)

if current_state.next:        # 如果 next 不为空，说明还有节点等待执行（即触发了 interrupt）
    # ... 获取用户输入 ...
    inputs = Command(resume=user_input)
    continue
```

- current_state.next

  ：一个元组，里面是接下来要执行的节点名称。

  - 如果 next 有值 → 图处于中断状态（human_review 节点暂停了）
  - 如果 next 为空 → 图已正常到达 END

- 这是目前检测是否触发 Human-in-the-Loop 最可靠的方式，比单纯依赖异常捕获更稳定。

**Command(resume=user_input)**

```Python
inputs = Command(resume=user_input)
```

- 用于**恢复**被 interrupt 暂停的图执行。
- resume= 后面的值会直接传递给 interrupt() 函数的返回值（即 feedback）。
- 每次恢复后，图会从 human_review 节点继续往下执行（先执行 router，再决定 revise 或 accept）。

**使用同一个 thread_config（含 thread_id）**

```Python
thread_config = {"configurable": {"thread_id": "hil_weather_v3"}}
```

- MemorySaver 需要通过 thread_id 来识别不同的对话线程。
- 整个交互过程中必须保持同一个 thread_id，才能让状态（消息历史、中断信息）正确恢复。

**review_router 的决策逻辑**

```Python
if any(k in last_content for k in ["yes", "批准", "ok", "good", "满意", "可以"]):
    return "accept"
```

- 根据用户最后输入的内容判断路由方向。
- 当用户输入 good 时，也会被识别为“接受”，从而结束流程（这也是你测试时输入 good 后成功结束的原因）。